# Evaluator — Quickstart Notebook

End-to-end walkthrough of the `evaluator` library for audio retrieval evaluation.

**Pipeline:** text question → TTS audio → ASR transcription → text embedding → corpus retrieval → IR metrics

**Sections:**
1. Install
2. Prepare sample data
3. Build config
4. Run evaluation
5. Inspect results
6. Compare retrieval modes
7. Audio embedding retrieval with FAISS
8. Grid search over retrieval params
9. Save / load results
10. LLM-as-judge scoring

## 1. Install

In [1]:
# Install the evaluator package from repo root
%pip install -q -e '..'

# datasets + soundfile needed to download PubMedQA sample data
%pip install -q datasets soundfile

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 2. Prepare sample data

Evaluator expects two JSON files:
- **`questions.json`** — queries with ground-truth doc IDs  
- **`corpus.json`** — document collection to retrieve from

We use [PubMedQA](https://huggingface.co/datasets/pubmed_qa) (medical Q&A) as a small realistic example.

In [3]:
from pathlib import Path
import json

DATA_DIR       = Path("data/pubmed_qa_small")
QUESTIONS_PATH = DATA_DIR / "questions.json"
CORPUS_PATH    = DATA_DIR / "corpus.json"


if QUESTIONS_PATH.exists() and CORPUS_PATH.exists():
    print("Data already present — skipping download.")
else:
    from datasets import load_dataset

    N_SAMPLES = 20
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    hf_ds = load_dataset("pubmed_qa", "pqa_labeled", split=f"train[:{N_SAMPLES}]")


    questions, corpus = [], []
    for row in hf_ds:
        doc_id  = str(row["pubid"])
        context = "\n\n".join(c.strip() for c in row["context"]["contexts"] if c.strip())
        if row.get("long_answer"):
            context = f"{context}\n\n{row['long_answer'].strip()}" if context else row["long_answer"].strip()

        corpus.append({
            "doc_id":   doc_id,
            "title":    row["question"],
            "text":     context,
            "language": "en",
            "metadata": {"pubid": row["pubid"], "decision": row.get("final_decision")},
        })
        questions.append({
            "question_id":         f"q_{doc_id}",
            "question_text":       row["question"],
            "groundtruth_doc_ids": [doc_id],
            "relevance_grades":    {doc_id: 1},
            "language":            "en",
            "metadata":            {"pubid": row["pubid"]},
        })

    QUESTIONS_PATH.write_text(json.dumps(questions, indent=2))
    CORPUS_PATH.write_text(json.dumps(corpus, indent=2))
    print(f"Saved {len(questions)} questions and {len(corpus)} corpus docs.")

questions = json.loads(QUESTIONS_PATH.read_text())
corpus    = json.loads(CORPUS_PATH.read_text())
print(f"Questions: {len(questions)}  |  Corpus docs: {len(corpus)}")
print("Sample question:", questions[0]["question_text"])

Data already present — skipping download.
Questions: 20  |  Corpus docs: 20
Sample question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?


## 3. Build config

`EvaluationConfig` is the central config object. Every aspect of the pipeline is controlled here — models, retrieval, audio synthesis, caching, devices.

In [4]:
from evaluator import EvaluationConfig

cfg = EvaluationConfig.from_dict({
    "experiment_name": "pubmed_qa_whisper_labse",
    "output_dir":      str(DATA_DIR / "results"),

    # --- Models ---
    "model": {
        # audio → ASR transcript → text embedding → retrieve
        "pipeline_mode":       "asr_text_retrieval",
        "asr_model_type":      "whisper",
        "asr_model_name":      "openai/whisper-tiny",
        "asr_device":          "cpu",
        "text_emb_model_type": "labse",
        "text_emb_model_name": "sentence-transformers/LaBSE",
        "text_emb_device":     "cpu",
    },

    # --- Data ---
    "data": {
        "questions_path":    str(QUESTIONS_PATH),
        "corpus_path":       str(CORPUS_PATH),
        "batch_size":        2,
        "trace_limit":       5,       # limit to first 5 queries for speed
        "strict_validation": False,
    },

    # --- Retrieval ---
    "vector_db": {
        "type":           "inmemory",  # faiss also available
        "k":              5,
        "retrieval_mode": "dense",     # dense | sparse | hybrid
    },

    # --- Audio synthesis (auto-runs if audio_path missing) ---
    "audio_synthesis": {
        "enabled":    True,
        "provider":   "mms",           # piper | mms | xtts_v2
        "language":   "en",
        "output_dir": str(DATA_DIR / "audio"),
    },

    # --- Caching (speeds up reruns) ---
    "cache": {
        "enabled":   True,
        "cache_dir": str(DATA_DIR / "cache"),
    },

    # --- Service runtime ---
    "service_runtime": {
        "startup_mode":   "lazy",       # load models on demand
        "offload_policy": "on_finish",  # free GPU memory after each stage
    },

    "tracking": {"enabled": False},
})

print(f"Pipeline:   {cfg.model.pipeline_mode}")
print(f"ASR:        {cfg.model.asr_model_name}")
print(f"Embedding:  {cfg.model.text_emb_model_name}")
print(f"Retrieval:  {cfg.vector_db.retrieval_mode}, k={cfg.vector_db.k}")
print(f"TTS:        {cfg.audio_synthesis.provider}")

Pipeline:   asr_text_retrieval
ASR:        openai/whisper-tiny
Embedding:  sentence-transformers/LaBSE
Retrieval:  dense, k=5
TTS:        mms


In [5]:
# Persist config to YAML for reproducibility
cfg.to_yaml(str(DATA_DIR / "config.yaml"))

# Reload and verify round-trip
cfg_rt = EvaluationConfig.from_yaml(str(DATA_DIR / "config.yaml"))
assert cfg_rt.experiment_name == cfg.experiment_name
print("Config saved and round-trip verified:", DATA_DIR / "config.yaml")

Config saved and round-trip verified: data/pubmed_qa_small/config.yaml


## 4. Run evaluation

Audio synthesis runs automatically for questions without `audio_path`.

In [6]:
from evaluator import api
results = api.run_evaluation(cfg)

INFO | evaluator | Logging initialized. Log file: logs/2026-04-25_15-06-52_pubmed_qa_whisper_labse.log
INFO | evaluator.services.model_services | service.start label=asr:whisper@cpu load_time=3.24s
INFO | evaluator.services.model_services | service.start label=text_embedding:labse@cpu load_time=4.84s
INFO | evaluator.pipeline.retrieval_pipeline | Retrieval pipeline initialized with vector store: InMemoryVectorStore
INFO | evaluator.services.evaluation_service | service.startup_policy mode=lazy
INFO | evaluator.services.evaluation_service | Audio synthesis enabled - checking for questions needing synthesis
INFO | evaluator.models.tts.mms_tts | MMS TTS initialized with model: facebook/mms-tts-eng
INFO | evaluator.pipeline.audio.synthesis | Initialized TTS model: mms
INFO | evaluator.services.evaluation_service | Synthesizing audio for 5 questions...
INFO | evaluator.services.evaluation_service | Audio synthesis complete for 5 questions
INFO | evaluator.services.evaluation_service | Loade

ASR Processing: 100%|██████████| 3/3 [00:01<00:00,  2.76it/s]

INFO | evaluator.pipeline.asr_pipeline | ASR Dataset Processing completed in 1.09s
INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 3 batches processed
INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ASR Phase complete: 5 transcriptions


INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 2: Text Embedding (from ASR)
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Embedding Phase completed in 0.02s
INFO | evaluator.evaluation.phased | Text Embedding Phase complete: 5 embeddings
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 3: Retrieval
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Retrieval Phase completed in 0.00s
INFO | evaluator.evaluation.phased | Retrieval Phase complete: 5 queries
INFO | evaluator.evaluation.phased | DB vectors shape: (20, 768)
INFO | evaluator.evaluation.phased | DB vectors stats - mean: -0.0070, std: 0.0354
INFO | evaluator.evaluation.phased | DB vectors norms - mean: 1.0000
INFO 

## 5. Inspect results

In [7]:
# One-line summary
print(results.summary())

# Key metrics
print(f"\nMRR:      {results.get_metric('MRR', 0.0):.4f}")
print(f"Recall@5: {results.get_metric('Recall@5', 0.0):.4f}")
print(f"WER:      {results.get_metric('WER', 0.0):.2%}")

MRR: 1.0000, MAP: 1.0000, WER: 45.66%, CER: 16.10%, Recall@5: 1.0000, Recall@10: 1.0000, NDCG@5: 1.0000, pipeline_mode: asr_text_retrieval, phased: True, asr: WhisperModel - openai/whisper-tiny, embedder: LaBseModel - sentence-transformers/LaBSE, Recall@1: 1.0000, NDCG@1: 1.0000, NDCG@10: 1.0000, query_traces: [{'query_id': 'q_21645374', 'relevant': {'21645374': 1}, 'retrieved': [{'doc_key': '21645374', 'score': 0.1616632491350174}, {'doc_key': '26037986', 'score': 0.06208791956305504}, {'doc_key': '11729377', 'score': 0.03331040218472481}, {'doc_key': '17096624', 'score': -0.005656194873154163}, {'doc_key': '25957366', 'score': -0.01102147065103054}]}, {'query_id': 'q_16418930', 'relevant': {'16418930': 1}, 'retrieved': [{'doc_key': '16418930', 'score': 0.10043227672576904}, {'doc_key': '17113061', 'score': 0.057011839002370834}, {'doc_key': '21645374', 'score': 0.03538094088435173}, {'doc_key': '22694248', 'score': 0.028737211599946022}, {'doc_key': '25432938', 'score': 0.01385646592

In [8]:
# All metrics
print(f"{'Metric':<22} {'Value'}")
print("-" * 35)
for name, value in sorted(results.metrics.items()):
    fmt = f"{value:.4f}" if isinstance(value, float) else str(value)
    print(f"{name:<22} {fmt}")

Metric                 Value
-----------------------------------
CER                    0.1610
MAP                    1.0000
MRR                    1.0000
NDCG@1                 1.0000
NDCG@10                1.0000
NDCG@5                 1.0000
Recall@1               1.0000
Recall@10              1.0000
Recall@5               1.0000
WER                    0.4566
asr                    WhisperModel - openai/whisper-tiny
embedder               LaBseModel - sentence-transformers/LaBSE
phased                 True
pipeline_mode          asr_text_retrieval
query_traces           [{'query_id': 'q_21645374', 'relevant': {'21645374': 1}, 'retrieved': [{'doc_key': '21645374', 'score': 0.1616632491350174}, {'doc_key': '26037986', 'score': 0.06208791956305504}, {'doc_key': '11729377', 'score': 0.03331040218472481}, {'doc_key': '17096624', 'score': -0.005656194873154163}, {'doc_key': '25957366', 'score': -0.01102147065103054}]}, {'query_id': 'q_16418930', 'relevant': {'16418930': 1}, 'retrieved': [

In [9]:
# Run metadata
meta = results.metadata
print(f"Duration:    {meta.get('duration_seconds', 0):.1f}s")
print(f"Num samples: {meta.get('num_samples', '?')}")
print(f"Created at:  {meta.get('created_at', '?')}")

Duration:    12.4s
Num samples: 5
Created at:  2026-04-25T15:07:04.551070


In [10]:
print(results)

Evaluation Results: pubmed_qa_whisper_labse
Pipeline Mode: asr_text_retrieval

Metrics:
  CER           : 16.10%
  MAP           : 1.00
  MRR           : 1.00
  NDCG@1        : 1.00
  NDCG@10       : 1.00
  NDCG@5        : 1.00
  Recall@1      : 1.00
  Recall@10     : 1.00
  Recall@5      : 1.00
  WER           : 45.66%
  asr           : WhisperModel - openai/whisper-tiny
  embedder      : LaBseModel - sentence-transformers/LaBSE
  phased        : True
  pipeline_mode : asr_text_retrieval
  query_traces  : [{'query_id': 'q_21645374', 'relevant': {'21645374': 1}, 'retrieved': [{'doc_key': '21645374', 'score': 0.1616632491350174}, {'doc_key': '26037986', 'score': 0.06208791956305504}, {'doc_key': '11729377', 'score': 0.03331040218472481}, {'doc_key': '17096624', 'score': -0.005656194873154163}, {'doc_key': '25957366', 'score': -0.01102147065103054}]}, {'query_id': 'q_16418930', 'relevant': {'16418930': 1}, 'retrieved': [{'doc_key': '16418930', 'score': 0.10043227672576904}, {'doc_key': '

## 6. Compare retrieval modes

Build variant configs with `dataclasses.replace` and run sequentially.

In [11]:
from evaluator import run_evaluation
from dataclasses import replace

cfg_dense = replace(
    cfg,
    experiment_name="dense",
    vector_db=replace(cfg.vector_db, retrieval_mode="dense"),
)

cfg_hybrid = replace(
    cfg,
    experiment_name="hybrid_rrf",
    vector_db=replace(
        cfg.vector_db,
        retrieval_mode="hybrid",
        hybrid_dense_weight=0.6,
        hybrid_fusion_method="rrf",
    ),
)

all_results = [run_evaluation(c) for c in [cfg_dense, cfg_hybrid]]

print(f"{'Experiment':<15} {'MRR':>8} {'Recall@5':>10} {'WER':>8}")
print("-" * 45)
for r in all_results:
    print(
        f"{r.config.experiment_name:<15}"
        f" {r.get_metric('MRR', 0.0):>8.4f}"
        f" {r.get_metric('Recall@5', 0.0):>10.4f}"
        f" {r.get_metric('WER', 0.0):>8.2%}"
    )

INFO | evaluator | Logging initialized. Log file: logs/2026-04-25_15-07-04_dense.log
INFO | evaluator.services.model_services | service.start label=asr:whisper@cpu load_time=2.92s
INFO | evaluator.services.model_services | service.start label=text_embedding:labse@cpu load_time=4.11s
INFO | evaluator.pipeline.retrieval_pipeline | Retrieval pipeline initialized with vector store: InMemoryVectorStore
INFO | evaluator.services.evaluation_service | service.startup_policy mode=lazy
INFO | evaluator.services.evaluation_service | Audio synthesis enabled - checking for questions needing synthesis
INFO | evaluator.models.tts.mms_tts | MMS TTS initialized with model: facebook/mms-tts-eng
INFO | evaluator.pipeline.audio.synthesis | Initialized TTS model: mms
INFO | evaluator.services.evaluation_service | Synthesizing audio for 5 questions...
INFO | evaluator.services.evaluation_service | Audio synthesis complete for 5 questions
INFO | evaluator.services.evaluation_service | Loaded cached retrieval

ASR Processing: 100%|██████████| 3/3 [00:00<00:00,  3.13it/s]

INFO | evaluator.pipeline.asr_pipeline | ASR Dataset Processing completed in 0.96s
INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 3 batches processed


INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ASR Phase complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 2: Text Embedding (from ASR)
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Embedding Phase completed in 0.02s
INFO | evaluator.evaluation.phased | Text Embedding Phase complete: 5 embeddings
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 3: Retrieval
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Retrieval Phase completed in 0.00s
INFO | evaluator.evaluation.phased | Retrieval Phase complete: 5 queries
INFO | evaluator.evaluation.phased | DB vectors shape: (20, 768)


ASR Processing: 100%|██████████| 3/3 [00:00<00:00,  3.30it/s]

INFO | evaluator.pipeline.asr_pipeline | ASR Dataset Processing completed in 0.91s
INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 3 batches processed


INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ASR Phase complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 2: Text Embedding (from ASR)
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Embedding Phase completed in 0.02s
INFO | evaluator.evaluation.phased | Text Embedding Phase complete: 5 embeddings
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 3: Retrieval
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Retrieval Phase completed in 0.00s
INFO | evaluator.evaluation.phased | Retrieval Phase complete: 5 queries
INFO | evaluator.evaluation.phased | DB vectors shape: (20, 768)


## 7. Audio embedding retrieval with FAISS

`audio_emb_retrieval` skips ASR entirely — raw audio is encoded directly into a vector via an audio embedding model (attention pooling over Whisper encoder), then retrieved against a FAISS index.

**When to use:** when you want to measure audio-side retrieval quality independently of ASR errors, or when your documents have audio representations rather than text.

In [12]:
from evaluator import EvaluationConfig, run_evaluation
from dataclasses import replace

cfg_audio = EvaluationConfig.from_dict({
    "experiment_name": "pubmed_qa_audio_emb_faiss",
    "output_dir":      str(DATA_DIR / "results"),

    "model": {
        # Raw audio → audio embedding → retrieve (no ASR step)
        "pipeline_mode":        "audio_emb_retrieval",
        "audio_emb_model_type": "attention_pool",          # whisper encoder + attention pooling

        # Use a HuggingFace model ID (downloaded on first run) OR a local path:
        "audio_emb_model_name": "openai/whisper-large",   # HuggingFace ID
        # "audio_emb_model_name": "/models/whisper-large", # local HF snapshot dir

        # Optional: path to fine-tuned attention-pool weights (.pt file).
        # If omitted, uses the raw Whisper encoder (no fine-tuning applied).
        "audio_emb_model_path": "/home/krystian/studia/magisterka/attention_pool_model/apm_jina_admed_voice.pt",

        "audio_emb_device":     "cuda:0",
        "audio_emb_dim":        2048,  # whisper-large hidden dim; match your weights
    },

    "data": {
        "questions_path":    str(QUESTIONS_PATH),
        "corpus_path":       str(CORPUS_PATH),
        "batch_size":        2,
        "trace_limit":       5,
        "strict_validation": False,
    },

    "vector_db": {
        "type": "faiss",   # FAISS flat inner-product index
        "k":    5,
    },

    # Audio already synthesized in section 4 — reuse same output dir
    "audio_synthesis": {
        "enabled":    True,
        "provider":   "mms",
        "language":   "en",
        "output_dir": str(DATA_DIR / "audio"),
    },

    "cache": {
        "enabled":   True,
        "cache_dir": str(DATA_DIR / "cache"),
    },

    "service_runtime": {
        "startup_mode":   "lazy",
        "offload_policy": "on_finish",
    },

    "tracking": {"enabled": False},
})

print(f"Pipeline:      {cfg_audio.model.pipeline_mode}")
print(f"Audio model:   {cfg_audio.model.audio_emb_model_name}")
print(f"Custom weights:{cfg_audio.model.audio_emb_model_path or '(none — raw encoder)'}")
print(f"Embedding dim: {cfg_audio.model.audio_emb_dim}")
print(f"Vector store:  {cfg_audio.vector_db.type}")

results_audio = run_evaluation(cfg_audio)

print(f"\nMRR:      {results_audio.get_metric('MRR', 0.0):.4f}")
print(f"Recall@5: {results_audio.get_metric('Recall@5', 0.0):.4f}")
print("\n--- vs ASR+text pipeline ---")
print(f"{'':20} {'audio_emb':>12} {'asr_text':>12}")
print(f"{'MRR':20} {results_audio.get_metric('MRR', 0.0):>12.4f} {results.get_metric('MRR', 0.0):>12.4f}")
print(f"{'Recall@5':20} {results_audio.get_metric('Recall@5', 0.0):>12.4f} {results.get_metric('Recall@5', 0.0):>12.4f}")

Pipeline:      audio_emb_retrieval
Audio model:   openai/whisper-large
Custom weights:/home/krystian/studia/magisterka/attention_pool_model/apm_jina_admed_voice.pt
Embedding dim: 2048
Vector store:  faiss
INFO | evaluator | Logging initialized. Log file: logs/2026-04-25_15-07-28_pubmed_qa_audio_emb_faiss.log
INFO | evaluator.services.model_services | service.start label=audio_embedding:attention_pool@cuda:0 load_time=3.24s
INFO | evaluator.pipeline.audio_embedding_pipeline | AudioEmbedding pipeline initialized with model: AttentionPoolAudioModel - encoder:openai/whisper-large - emb_dim:2048 - weights:/home/krystian/studia/magisterka/attention_pool_model/apm_jina_admed_voice.pt
INFO | evaluator.pipeline.retrieval_pipeline | Retrieval pipeline initialized with vector store: FaissVectorStore
INFO | evaluator.services.evaluation_service | service.startup_policy mode=lazy
INFO | evaluator.services.evaluation_service | Audio synthesis enabled - checking for questions needing synthesis
INFO |

Audio embedding:   0%|          | 0/3 [00:00<?, ?it/s]

INFO | evaluator.pipeline.audio_embedding_pipeline | Audio embedding cache: 2/2 hits
INFO | evaluator.pipeline.audio_embedding_pipeline | Audio embedding cache: 2/2 hits
INFO | evaluator.pipeline.audio_embedding_pipeline | Audio embedding cache: 1/1 hits


Audio embedding: 100%|██████████| 3/3 [00:00<00:00, 44.00it/s]

INFO | evaluator.evaluation.phased | Audio Embedding Phase completed in 0.07s
INFO | evaluator.evaluation.phased | Audio Embedding Phase complete: 5 audio embeddings
INFO | evaluator.evaluation.phased | Audio embedding shape: (5, 2048)
INFO | evaluator.evaluation.phased | Audio embedding stats - mean: -0.0006, std: 0.0221
INFO | evaluator.evaluation.phased | Audio embedding norms - mean: 1.0000
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 3: Retrieval
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Retrieval Phase completed in 0.00s
INFO | evaluator.evaluation.phased | Retrieval Phase complete: 5 queries
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | RETRIEVAL DEBUG SAMPLE (first 3 queries):
INFO | evaluator.evaluation.phased | =====================

INFO | evaluator.services.model_services | service.stop label=audio_embedding:attention_pool@cuda:0

MRR:      0.5400
Recall@5: 0.8000

--- vs ASR+text pipeline ---
                        audio_emb     asr_text
MRR                        0.5400       1.0000
Recall@5                   0.8000       1.0000


## 8. Grid search over retrieval params

`GridSearch` sweeps config parameters systematically.

In [13]:
from evaluator.analysis.grid_search import GridSearch, run_grid_search, analyze_grid_results

grid = GridSearch(cfg)
grid.add_param("vector_db.k", [3, 5, 10])

print(f"Grid size: {grid.get_size()} configurations")

Grid size: 3 configurations


In [14]:
from evaluator import run_evaluation

grid_results = run_grid_search(grid, lambda c: run_evaluation(c).metrics)

analysis = analyze_grid_results(grid_results, metric_name="MRR", top_k=3)

print(f"Best params: {analysis['best_config']['params']}")
print(f"Best MRR:    {analysis['best_config']['MRR']:.4f}")
print()
for entry in analysis["top_configs"]:
    print(f"  #{entry['rank']}  k={entry['params']['vector_db.k']}  MRR={entry['MRR']:.4f}")

stats = analysis["summary"]["metric_stats"]
print(f"\nMRR range: {stats['min']:.4f} – {stats['max']:.4f}")

INFO | evaluator.analysis.grid_search | Starting grid search with 3 configurations
INFO | evaluator.analysis.grid_search | Evaluating configuration 1/3: {'vector_db.k': 3}
INFO | evaluator | Logging initialized. Log file: logs/2026-04-25_15-07-36_pubmed_qa_whisper_labse.log
INFO | evaluator.services.model_services | service.start label=asr:whisper@cpu load_time=3.42s
INFO | evaluator.services.model_services | service.start label=text_embedding:labse@cpu load_time=4.80s
INFO | evaluator.pipeline.retrieval_pipeline | Retrieval pipeline initialized with vector store: InMemoryVectorStore
INFO | evaluator.services.evaluation_service | service.startup_policy mode=lazy
INFO | evaluator.services.evaluation_service | Audio synthesis enabled - checking for questions needing synthesis
INFO | evaluator.models.tts.mms_tts | MMS TTS initialized with model: facebook/mms-tts-eng
INFO | evaluator.pipeline.audio.synthesis | Initialized TTS model: mms
INFO | evaluator.services.evaluation_service | Synthe

ASR Processing: 100%|██████████| 3/3 [00:01<00:00,  3.00it/s]

INFO | evaluator.pipeline.asr_pipeline | ASR Dataset Processing completed in 1.00s
INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 3 batches processed
INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ASR Phase complete: 5 transcriptions


INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 2: Text Embedding (from ASR)
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Embedding Phase completed in 0.02s
INFO | evaluator.evaluation.phased | Text Embedding Phase complete: 5 embeddings
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 3: Retrieval
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Retrieval Phase completed in 0.00s
INFO | evaluator.evaluation.phased | Retrieval Phase complete: 5 queries
INFO | evaluator.evaluation.phased | DB vectors shape: (20, 768)
INFO | evaluator.evaluation.phased | DB vectors stats - mean: -0.0070, std: 0.0354
INFO | evaluator.evaluation.phased | DB vectors norms - mean: 1.0000
INFO 

ASR Processing: 100%|██████████| 3/3 [00:00<00:00,  3.21it/s]

INFO | evaluator.pipeline.asr_pipeline | ASR Dataset Processing completed in 0.94s
INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 3 batches processed


INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ASR Phase complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 2: Text Embedding (from ASR)
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Embedding Phase completed in 0.03s
INFO | evaluator.evaluation.phased | Text Embedding Phase complete: 5 embeddings
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 3: Retrieval
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Retrieval Phase completed in 0.00s
INFO | evaluator.evaluation.phased | Retrieval Phase complete: 5 queries
INFO | evaluator.evaluation.phased | DB vectors shape: (20, 768)


ASR Processing: 100%|██████████| 3/3 [00:00<00:00,  3.32it/s]

INFO | evaluator.pipeline.asr_pipeline | ASR Dataset Processing completed in 0.90s
INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 3 batches processed


INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ASR Phase complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 2: Text Embedding (from ASR)
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Embedding Phase completed in 0.02s
INFO | evaluator.evaluation.phased | Text Embedding Phase complete: 5 embeddings
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 3: Retrieval
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Retrieval Phase completed in 0.00s
INFO | evaluator.evaluation.phased | Retrieval Phase complete: 5 queries
INFO | evaluator.evaluation.phased | DB vectors shape: (20, 768)


## 9. Save and load results

In [15]:
from evaluator import EvaluationResults

RESULTS_PATH = DATA_DIR / "results" / "run.json"

results.save(str(RESULTS_PATH))
print(f"Saved to {RESULTS_PATH}")

loaded = EvaluationResults.load(str(RESULTS_PATH))
assert loaded.metrics == results.metrics
print("Round-trip OK:", loaded.summary())

Saved to data/pubmed_qa_small/results/run.json
Round-trip OK: MRR: 1.0000, MAP: 1.0000, WER: 45.66%, CER: 16.10%, Recall@5: 1.0000, Recall@10: 1.0000, NDCG@5: 1.0000, pipeline_mode: asr_text_retrieval, phased: True, asr: WhisperModel - openai/whisper-tiny, embedder: LaBseModel - sentence-transformers/LaBSE, Recall@1: 1.0000, NDCG@1: 1.0000, NDCG@10: 1.0000, query_traces: [{'query_id': 'q_21645374', 'relevant': {'21645374': 1}, 'retrieved': [{'doc_key': '21645374', 'score': 0.1616632491350174}, {'doc_key': '26037986', 'score': 0.06208791956305504}, {'doc_key': '11729377', 'score': 0.03331040218472481}, {'doc_key': '17096624', 'score': -0.005656194873154163}, {'doc_key': '25957366', 'score': -0.01102147065103054}]}, {'query_id': 'q_16418930', 'relevant': {'16418930': 1}, 'retrieved': [{'doc_key': '16418930', 'score': 0.10043227672576904}, {'doc_key': '17113061', 'score': 0.057011839002370834}, {'doc_key': '21645374', 'score': 0.03538094088435173}, {'doc_key': '22694248', 'score': 0.02873

## 10. LLM-as-judge scoring

IR metrics (MRR, Recall@k) measure whether the ground-truth doc ID was retrieved, but they can't tell you *how relevant* the retrieved documents actually are. LLM-as-judge closes that gap by asking a language model to score each retrieval result for relevance, accuracy, and other aspects.

Two usage patterns:
- **Via config** — set `judge.enabled = True` before running; judge runs automatically and stores results in `results.metrics["llm_judge"]`
- **Post-hoc** — call `run_llm_judging` directly on `query_traces` from any results object

Any OpenAI-compatible endpoint works (OpenAI, Anthropic, local Ollama, vLLM, etc.).

### Pattern A — judge via config (judge runs as part of evaluation)

In [16]:
import os, json, urllib.request
from dataclasses import replace
from evaluator import run_evaluation

os.environ.setdefault("LOCAL_JUDGE_KEY", "ollama")

# Preferred model name — set to whatever you'd like to use.
# Will fuzzy-match against models actually available in Ollama.
PREFERRED_JUDGE_MODEL = "gemma"


def _pick_ollama_model(preferred: str, available: list[str]) -> tuple[str, bool]:
    """Return (chosen_name, exact_match)."""
    if not available:
        return preferred, False
    # 1. exact
    if preferred in available:
        return preferred, True
    # 2. prefix / substring (case-insensitive, strip tag)
    pref_base = preferred.split(":")[0].lower()
    for name in available:
        if name.split(":")[0].lower() == pref_base:
            return name, False
    for name in available:
        if pref_base in name.lower() or name.split(":")[0].lower() in pref_base:
            return name, False
    # 3. fallback: first model
    return available[0], False


_judge_available = False
_ollama_model = None
try:
    with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2) as r:
        tags = json.loads(r.read())
    _available_models = [m["name"] for m in tags.get("models", [])]
    if _available_models:
        _ollama_model, _exact = _pick_ollama_model(PREFERRED_JUDGE_MODEL, _available_models)
        _judge_available = True
        if not _exact:
            print(f"[judge] Requested '{PREFERRED_JUDGE_MODEL}' not found.")
            print(f"[judge] Available: {_available_models}")
            print(f"[judge] Using:     '{_ollama_model}'")
        else:
            print(f"[judge] Using '{_ollama_model}'")
except Exception:
    pass

if not _judge_available:
    print("Ollama not running or no models pulled.")
    print("Start:       ollama serve")
    print("Pull model:  ollama pull mistral  (or llama3.1, phi3, etc.)")
else:
    cfg_judged = replace(
        cfg,
        experiment_name="pubmed_qa_with_judge",
        judge=replace(
            cfg.judge,
            enabled=True,
            use_local_server=True,
            local_server_url="http://localhost:11434/v1/chat/completions",
            model=_ollama_model,
            api_key_env="LOCAL_JUDGE_KEY",
            temperature=0.0,
            max_cases=5,

            # --- Cloud alternative ---
            # use_local_server=False,
            # api_base="https://api.openai.com/v1/chat/completions",
            # model="gpt-4o-mini",
            # api_key_env="OPENAI_API_KEY",
        ),
    )

    results_judged = run_evaluation(cfg_judged)
    judge_out = results_judged.get_metric("llm_judge")
    print(f"\nJudge model:  {judge_out['model']}")
    print(f"Cases scored: {judge_out['cases']}")
    print(f"Mean score:   {judge_out['mean_score']:.3f}  (0–1 scale)")
    print()
    for row in judge_out["details"]:
        j = row["judge"]
        print(f"  {row['query_id']:20s}  score={j['score']:.2f}  [{j['verdict']}]  {j['reason'][:60]}")

[judge] Requested 'gemma' not found.
[judge] Available: ['gemma4:latest', 'qwen3.5:latest', 'mistral:7b-instruct', 'qwen3-coder:30b']
[judge] Using:     'gemma4:latest'
INFO | evaluator | Logging initialized. Log file: logs/2026-04-25_15-08-11_pubmed_qa_with_judge.log


INFO | evaluator.services.model_services | service.start label=asr:whisper@cpu load_time=2.93s
INFO | evaluator.services.model_services | service.start label=text_embedding:labse@cpu load_time=4.05s
INFO | evaluator.pipeline.retrieval_pipeline | Retrieval pipeline initialized with vector store: InMemoryVectorStore
INFO | evaluator.services.evaluation_service | service.startup_policy mode=lazy
INFO | evaluator.services.model_services | service.reuse label=llm_server:ollama@localhost:11434
INFO | evaluator.services.evaluation_service | llm.local_runtime_ready backend=ollama model=mistral:7b-instruct api_url=http://localhost:11434/v1/chat/completions
INFO | evaluator.services.evaluation_service | Audio synthesis enabled - checking for questions needing synthesis
INFO | evaluator.models.tts.mms_tts | MMS TTS initialized with model: facebook/mms-tts-eng
INFO | evaluator.pipeline.audio.synthesis | Initialized TTS model: mms
INFO | evaluator.services.evaluation_service | Synthesizing audio fo

ASR Processing: 100%|██████████| 3/3 [00:00<00:00,  3.09it/s]

INFO | evaluator.pipeline.asr_pipeline | ASR Dataset Processing completed in 0.97s
INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 3 batches processed


INFO | evaluator.pipeline.asr_pipeline | ASR processing complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ASR Phase complete: 5 transcriptions
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 2: Text Embedding (from ASR)
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Embedding Phase completed in 0.02s
INFO | evaluator.evaluation.phased | Text Embedding Phase complete: 5 embeddings
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | PHASE 3: Retrieval
INFO | evaluator.evaluation.phased | ==================================================
INFO | evaluator.evaluation.phased | Retrieval Phase completed in 0.00s
INFO | evaluator.evaluation.phased | Retrieval Phase complete: 5 queries
INFO | evaluator.evaluation.phased | DB vectors shape: (20, 768)


### Pattern B — post-hoc judging on existing traces

Use `run_llm_judging` directly when you already have results and want to judge without re-running the full pipeline.

In [17]:
from evaluator.judge import run_llm_judging

traces = results.get_metric("query_traces", [])
print(f"Available traces: {len(traces)}")
if traces:
    print("Sample trace keys:", list(traces[0].keys()))

if not _judge_available:
    print("\nSkipping post-hoc judge: Ollama not running.")
else:
    judge_results = run_llm_judging(
        traces,
        api_base="http://localhost:11434/v1/chat/completions",
        model=_ollama_model,
        api_key_env="LOCAL_JUDGE_KEY",
        temperature=0.0,
        max_cases=3,
        timeout_s=120,
    )
    print(f"\nPost-hoc judge — mean score: {judge_results['mean_score']:.3f}")
    for row in judge_results["details"]:
        j = row["judge"]
        print(f"  {row['query_id']}  {j['verdict']}  {j['score']:.2f}")

Available traces: 5
Sample trace keys: ['query_id', 'relevant', 'retrieved']

Post-hoc judge — mean score: 0.330
  q_21645374  PASS  0.33
  q_16418930  PASS  0.33
  q_9488747  FAIL  0.33


### Pattern C — multi-aspect scoring on a single result

`judge_multi_aspect` lets you score one retrieved document across custom dimensions (relevance, accuracy, completeness, etc.).

In [18]:
from evaluator.judge import judge_multi_aspect

sample_trace = traces[0] if traces else None

if sample_trace and _judge_available:
    query_id = sample_trace["query_id"]
    top_doc_key = sample_trace["retrieved"][0]["doc_key"] if sample_trace["retrieved"] else None
    top_doc = next((d for d in corpus if str(d["doc_id"]) == top_doc_key), None)

    if top_doc:
        query_text = next(
            (q["question_text"] for q in questions if f"q_{top_doc_key}" == query_id),
            query_id,
        )

        verdict = judge_multi_aspect(
            query=query_text,
            retrieved_text=top_doc["text"][:800],
            aspects=["relevance", "accuracy", "completeness"],
            api_base="http://localhost:11434/v1/chat/completions",
            model=_ollama_model,
            api_key=os.getenv("LOCAL_JUDGE_KEY", "ollama"),
            temperature=0.0,
            timeout_s=120,
            chain_of_thought=True,
        )
        print(f"Query:   {query_text[:80]}...")
        print(f"Doc:     {top_doc_key}")
        print(f"Scores:  {verdict['aspect_scores']}")
        print(f"Overall: {verdict['overall_score']:.2f}")
        if verdict.get("raw_response"):
            print(f"\nReasoning:\n{verdict['raw_response'][:400]}")
    else:
        print(f"Doc {top_doc_key} not in corpus.")
elif not traces:
    print("No traces — re-run section 4 with trace_limit > 0.")
else:
    print("Skipping: Ollama not running.")

Query:   Do mitochondria play a role in remodelling lace plant leaves during programmed c...
Doc:     21645374
Scores:  {'relevance': 5.0, 'accuracy': 5.0, 'completeness': 3.0}
Overall: 4.33

Reasoning:
Relevance: The document directly addresses all key components of the query: the lace plant (*Aponogeton madagascariensis*), programmed cell death (PCD), and the role of mitochondria. It sets up the context for investigating this exact relationship. Score: 5
Accuracy: The information provided regarding PCD in the lace plant and the general gap in knowledge about mitochondrial roles in plant PCD app
